# Cicero Letters – Minimal Notebook for Exploration

Dieses minimal gehaltene Notebook dient als Experimentierumgebung für:

- **TF-IDF**
- **gensim Topic Modelling (LDA)**

Es arbeitet direkt mit bereits analysierten **`.conllu`-Dateien** und ist bewusst einfacher als die grosse Pipeline mit LatinCy und BERTopic.

## Lernziele

Nach diesem Notebook sollt ihr:

1. lemma-basierte Texte aus `.conllu` laden können
2. TF-IDF für Briefe berechnen können
3. ähnliche Briefe identifizieren können
4. mit **gensim LDA** Topics erzeugen und interpretieren können
5. die Auswirkungen von Parametern reflektieren können

## Hinweis

Dieses Notebook ist **explorativ**. Die Ergebnisse sind interpretierbar, aber nicht automatisch „wahr“. Gerade bei Topic Modelling solltet ihr immer prüfen, ob die Topics historisch sinnvoll sind.

In [ ]:
# Falls nötig, einmalig installieren:
# !pip install conllu gensim pyLDAvis networkx plotly

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from conllu import parse

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

import plotly.express as px
import plotly.graph_objects as go
import networkx as nx

import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

## 1. Daten laden

Das Notebook liest alle `.conllu`-Dateien aus einem Ordner und extrahiert die **Lemmata**.
Optional werden nur **Inhaltswörter** behalten:

- `NOUN`
- `PROPN`
- `ADJ`
- `VERB`

In [ ]:
conllu_dir = Path("outputs/conllu_letters")  # ggf. anpassen
files = sorted(conllu_dir.glob("*.conllu"))

print(f"Gefundene Dateien: {len(files)}")

ALLOWED_UPOS = {"NOUN", "PROPN", "ADJ", "VERB"}
USE_CONTENT_WORDS_ONLY = True
MIN_LEMMA_LEN = 2

LATIN_STOPWORDS = {
    "et", "in", "de", "ad", "non", "ut", "cum", "qui", "quae", "quod", "est", "esse",
    "sum", "ego", "tu", "nos", "vos", "hic", "ille", "is", "ea", "id", "autem", "enim",
    "sed", "si", "ne", "nec", "nam", "ita", "iam", "me", "te", "se", "mihi", "tibi",
    "sibi", "noster", "vester", "suus", "quid", "quoniam", "quoque", "tam", "tamen",
    "apud", "ab", "a", "ex", "e", "per", "pro", "post", "ante", "inter", "sine",
    "scribo", "littera",
    "puto", "uenio", "ago", "dico", "do", "dies", "magnus", "ualeo", "scio", "mitto" # weiter hinzugefügte Stoppwörter (korpusspezifisch)
}

rows = []

for file in files:
    content = file.read_text(encoding="utf-8")
    try:
        sentences = parse(content)
    except Exception as e:
        print(f"Fehler bei {file.name}: {e}")
        continue

    forms = []
    lemmas = []

    for sent in sentences:
        for tok in sent:
            if not isinstance(tok["id"], int):
                continue

            form = tok.get("form")
            lemma = tok.get("lemma")
            upos = tok.get("upostag") or tok.get("upos")

            if form:
                forms.append(form)

            if lemma:
                lemma = str(lemma).lower().strip()
                lemma = re.sub(r"[^a-zA-Zāēīōūȳăĕĭŏŭ]+", "", lemma)
                if len(lemma) < MIN_LEMMA_LEN:
                    continue
                if lemma in LATIN_STOPWORDS:
                    continue
                if USE_CONTENT_WORDS_ONLY and upos not in ALLOWED_UPOS:
                    continue
                lemmas.append(lemma)

    rows.append({
        "doc_id": file.stem,
        "raw_text": " ".join(forms),
        "clean_text": " ".join(lemmas),
        "n_tokens": len(forms),
        "n_terms": len(lemmas)
    })

df = pd.DataFrame(rows)
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

print(df.shape)
df.head()

## 2. Überblick über das Korpus

In [ ]:
display(df[["n_tokens", "n_terms"]].describe())

plt.figure(figsize=(8, 4))
plt.hist(df["n_terms"], bins=30)
plt.title("Verteilung der Dokumentlängen (bereinigte Terms)")
plt.xlabel("Anzahl Terms")
plt.ylabel("Anzahl Briefe")
plt.show()

## 3. TF-IDF

TF-IDF hilft dabei, Wörter zu identifizieren, die für einzelne Briefe oder für das Korpus charakteristisch sind.

In [ ]:
vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b",
    min_df=3,
    max_df=0.6,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["clean_text"])
feature_names = vectorizer.get_feature_names_out()

print("TF-IDF-Matrix:", X.shape)

In [ ]:
def top_tfidf_words(doc_index, n=10):
    row = X[doc_index].toarray().flatten()
    top_ids = row.argsort()[::-1][:n]
    return pd.DataFrame({
        "term": [feature_names[i] for i in top_ids],
        "score": [row[i] for i in top_ids]
    })

doc_index = 0
print("Brief:", df.loc[doc_index, "doc_id"])
top_tfidf_words(doc_index, 10)

### 3.1 Top-Wörter eines ausgewählten Briefs

In [ ]:
doc_index = 0  # hier ändern

top_doc = top_tfidf_words(doc_index, 10)

plt.figure(figsize=(8, 4))
plt.barh(top_doc["term"][::-1], top_doc["score"][::-1])
plt.title(f"Top TF-IDF Wörter: {df.loc[doc_index, 'doc_id']}")
plt.xlabel("TF-IDF")
plt.show()

### 3.2 Wichtigste TF-IDF-Wörter im gesamten Korpus

In [ ]:
tfidf_sum = np.asarray(X.sum(axis=0)).flatten()
top_ids = tfidf_sum.argsort()[::-1][:20]

top_corpus = pd.DataFrame({
    "term": [feature_names[i] for i in top_ids],
    "score": [tfidf_sum[i] for i in top_ids]
})

display(top_corpus)

plt.figure(figsize=(10, 5))
plt.barh(top_corpus["term"][::-1], top_corpus["score"][::-1])
plt.title("Wichtigste TF-IDF-Wörter im Korpus")
plt.xlabel("Aggregierter TF-IDF-Wert")
plt.show()

### 3.3 Ähnliche Briefe (Cosine Similarity)

In [ ]:
sim_matrix = cosine_similarity(X)
sim_df = pd.DataFrame(sim_matrix, index=df["doc_id"], columns=df["doc_id"])

target_doc = df.loc[0, "doc_id"]  # hier ändern
similar_docs = sim_df.loc[target_doc].sort_values(ascending=False)[1:11]

similar_docs

In [ ]:
target_doc = df.loc[0, "doc_id"]  # hier ändern
top_sim = sim_df.loc[target_doc].sort_values(ascending=False)[1:11].reset_index()
top_sim.columns = ["doc_id", "similarity"]

fig = px.bar(
    top_sim,
    x="similarity",
    y="doc_id",
    orientation="h",
    title=f"Ähnlichste Briefe zu {target_doc}"
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

### 3.4 Interaktive Similarity-Heatmap (Ausschnitt)

Für grosse Korpora ist die vollständige Matrix schwer lesbar. Daher wird hier nur ein Ausschnitt angezeigt.

In [ ]:
max_docs = min(30, len(df))
heat_df = sim_df.iloc[:max_docs, :max_docs]

fig = px.imshow(
    heat_df,
    aspect="auto",
    title=f"Cosine Similarity Heatmap (erste {max_docs} Briefe)",
    color_continuous_scale="Viridis"
)
fig.update_layout(width=850, height=750)
fig.show()

### 3.5 Interaktives Netzwerk ähnlicher Briefe

In [ ]:
threshold = 0.20

G = nx.Graph()
for doc_id in df["doc_id"]:
    G.add_node(doc_id)

n = len(df)
for i in range(n):
    for j in range(i + 1, n):
        sim = sim_matrix[i, j]
        if sim >= threshold:
            G.add_edge(df.loc[i, "doc_id"], df.loc[j, "doc_id"], weight=float(sim))

G_plot = G.copy()
isolates = list(nx.isolates(G_plot))
G_plot.remove_nodes_from(isolates)

print(f"Knoten: {G_plot.number_of_nodes()}, Kanten: {G_plot.number_of_edges()}")

pos = nx.spring_layout(G_plot, seed=42, k=0.35)

edge_x, edge_y = [], []
for u, v, data in G_plot.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    line=dict(width=0.7, color="#999"),
    hoverinfo="none",
    mode="lines"
)

degrees = dict(G_plot.degree())
node_x, node_y, node_text, node_size, node_color = [], [], [], [], []

for node in G_plot.nodes():
    x, y = pos[node]
    deg = degrees[node]
    node_x.append(x)
    node_y.append(y)
    node_size.append(10 + deg * 3)
    node_color.append(deg)
    node_text.append(f"<b>{node}</b><br>Degree: {deg}")

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers",
    hoverinfo="text",
    hovertext=node_text,
    marker=dict(
        showscale=True,
        colorscale="Viridis",
        color=node_color,
        size=node_size,
        colorbar=dict(title="Degree"),
        line_width=1
    )
)

fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title="Interaktives Netzwerk ähnlicher Briefe",
        title_x=0.5,
        showlegend=False,
        hovermode="closest",
        margin=dict(b=20, l=20, r=20, t=50),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=800
    )
)
fig.show()

## 4. Vorbereitung für gensim Topic Modelling

In [ ]:
texts = [doc.split() for doc in df["clean_text"]]

dictionary = corpora.Dictionary(texts)
dictionary.filter_extremes(no_below=3, no_above=0.6)

corpus = [dictionary.doc2bow(text) for text in texts]

print("Dokumente:", len(corpus))
print("Vokabular:", len(dictionary))

## 5. Topic Modelling mit gensim (LDA)

In [ ]:
NUM_TOPICS = 20  # hier experimentieren

lda = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    passes=15,
    iterations=300,
    random_state=42
)

In [ ]:
for i, topic in lda.print_topics(num_topics=NUM_TOPICS, num_words=10):
    print(f"Topic {i}: {topic}")

### 5.1 Kohärenz berechnen

In [ ]:
coherence_model = CoherenceModel(
    model=lda,
    texts=texts,
    dictionary=dictionary,
    coherence="c_v"
)

coherence = coherence_model.get_coherence()
print(f"Coherence (c_v): {coherence:.4f}")

### 5.2 Dominantes Topic pro Brief

In [ ]:
def dominant_topic_for_bow(bow):
    topic_probs = lda.get_document_topics(bow, minimum_probability=0.0)
    dominant_topic, dominant_prob = max(topic_probs, key=lambda x: x[1])
    return dominant_topic, dominant_prob

dominant = [dominant_topic_for_bow(bow) for bow in corpus]

df["dominant_topic"] = [x[0] for x in dominant]
df["topic_prob"] = [x[1] for x in dominant]

df[["doc_id", "dominant_topic", "topic_prob"]].head()

### 5.3 Topic-Verteilung visualisieren

In [ ]:
topic_counts = df["dominant_topic"].value_counts().sort_index()

plt.figure(figsize=(10, 4))
plt.bar(topic_counts.index.astype(str), topic_counts.values)
plt.title("Verteilung dominanter Topics")
plt.xlabel("Topic")
plt.ylabel("Anzahl Briefe")
plt.show()

In [ ]:
topic_counts_df = topic_counts.reset_index()
topic_counts_df.columns = ["topic", "count"]

fig = px.bar(
    topic_counts_df,
    x="topic",
    y="count",
    title="Verteilung dominanter Topics (interaktiv)"
)
fig.show()

### 5.4 Top-Wörter pro Topic als Balkendiagramme

In [ ]:
top_n = 10
topic_word_rows = []

for topic_id in range(NUM_TOPICS):
    words = lda.show_topic(topic_id, topn=top_n)
    for rank, (word, weight) in enumerate(words, start=1):
        topic_word_rows.append({
            "topic": topic_id,
            "rank": rank,
            "word": word,
            "weight": weight
        })

topic_words_df = pd.DataFrame(topic_word_rows)
topic_words_df.head(20)

In [ ]:
ncols = 2
nrows = int(np.ceil(NUM_TOPICS / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = np.array(axes).reshape(-1)

for topic_id in range(NUM_TOPICS):
    ax = axes[topic_id]
    sub = topic_words_df[topic_words_df["topic"] == topic_id].sort_values("weight", ascending=True)
    ax.barh(sub["word"], sub["weight"])
    ax.set_title(f"Topic {topic_id}")
    ax.set_xlabel("Gewicht")

for j in range(NUM_TOPICS, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

### 5.5 Dokument-Topic-Matrix

In [ ]:
doc_topic_matrix = []

for bow in corpus:
    topic_probs = lda.get_document_topics(bow, minimum_probability=0.0)
    doc_topic_matrix.append([prob for _, prob in topic_probs])

doc_topic_matrix = np.array(doc_topic_matrix)

topic_cols = [f"topic_{i}" for i in range(NUM_TOPICS)]
doc_topic_df = pd.DataFrame(doc_topic_matrix, columns=topic_cols)
doc_topic_df.insert(0, "doc_id", df["doc_id"].values)

doc_topic_df.head()

In [ ]:
max_docs = min(50, len(doc_topic_df))
heat_data = doc_topic_df.iloc[:max_docs, 1:]

fig = px.imshow(
    heat_data,
    aspect="auto",
    title=f"Dokument-Topic-Heatmap (erste {max_docs} Briefe)",
    color_continuous_scale="Viridis"
)
fig.update_layout(width=900, height=800)
fig.show()

### 5.6 Topic Similarity Heatmap

In [ ]:
topic_word_matrix = lda.get_topics()
topic_similarity = cosine_similarity(topic_word_matrix)

fig = px.imshow(
    topic_similarity,
    text_auto=True,
    title="Ähnlichkeit zwischen Topics",
    color_continuous_scale="Viridis"
)
fig.update_layout(width=700, height=650)
fig.show()

### 5.7 Interaktive pyLDAvis-Visualisierung

In [ ]:
vis = gensimvis.prepare(lda, corpus, dictionary)
pyLDAvis.display(vis)

## 6. Briefe eines bestimmten Topics lesen

Diese Zelle hilft euch, ein Topic historisch zu interpretieren.

In [ ]:
selected_topic = 0  # hier ändern

df[df["dominant_topic"] == selected_topic][["doc_id", "topic_prob", "raw_text"]].sort_values(
    "topic_prob", ascending=False
).head(5)

## 7. Dominante Topics exportieren

In [ ]:
# Beispiel: dominant topic pro Dokument bestimmen
df["dominant_topic"] = [dominant_topic_for_bow(bow)[0] for bow in corpus]

# OPTIONAL: Keywords pro Topic (falls vorhanden)
topic_keywords = {
    topic_id: ", ".join([word for word, _ in lda.show_topic(topic_id, topn=10)])
    for topic_id in range(lda.num_topics)
}

df["topic_keywords"] = df["dominant_topic"].map(topic_keywords)

# Export für Kapitel 5
df.to_csv("outputs/topic_modeling/letters_with_topics.csv", index=False)

## 8. Experimentieraufgaben

Probiert mindestens drei Dinge aus:

1. Ändert `NUM_TOPICS` (z. B. 5, 8, 10, 15)
2. Ändert `min_df` und `max_df` im `TfidfVectorizer`
3. Ändert `no_below` und `no_above` im `dictionary.filter_extremes`

Und beantwortet dabei:

- Welche Wörter erscheinen in den TF-IDF-Listen?
- Welche Topics sind historisch sinnvoll?
- Welche Einstellungen liefern interpretierbarere Ergebnisse?
- Wo stossen die Methoden an Grenzen?